# Opsætning: ecoinvent

Denne notebook sætter dit Brightway-projekt op med ecoinvent som baggrundsdatabase.

Den er lavet så du **kun downloader hvis du er nødt til det**. Hvis ecoinvent allerede
findes på din maskine, springer den downloaden over — så du ikke bruger 30 minutter på
noget du allerede har.

**Før du kører:** Kernel → Change Kernel → **Python (bw25)**

---

## Før du starter — to ting der skal være på plads

**1. Du skal have en ecoinvent-licens med dit eget brugernavn og kodeord.**
Login gennem universitetets portal (SSO) er *ikke* det samme. Hvis du kun kan tilgå
ecoinvent gennem en portal, har du måske ikke en konto der virker fra Python — spørg dit
bibliotek eller din vejleder.

**2. Du skal have logget ind på [ecoinvent.org](https://ecoinvent.org) i en browser
mindst én gang og accepteret licensaftalen og persondata-aftalen.**

> Dette er den fejl der koster mest tid. En helt ny konto der aldrig har været logget
> ind i browseren **fejler fra Python**, selvom brugernavn og kodeord er korrekte — og
> fejlbeskeden nævner ikke aftalerne med ét ord.

Kan du ikke logge ind på hjemmesiden, hjælper intet i Python.

## 1. Tjek at du kører den rigtige Python

Den hyppigste fejl af alle er `ModuleNotFoundError: No module named 'bw2data'`. Pakken
*er* installeret — notebooken kører bare en anden Python end den du installerede i.

Stien nedenfor skal indeholde `.venv`. Gør den ikke det: **Kernel → Change Kernel →
Python (bw25)**. Geninstallér ikke — det installerer bare samme pakke det samme forkerte
sted igen.

In [ ]:
import sys
print(sys.executable)

## 2. Vælg dit projekt

Et **projekt** i Brightway er et lukket arbejdsrum. Alt ligger inde i det: din egen
model, ecoinvent, biosfæren, metoderne. Projekter kan ikke se hinanden, og du er altid
i præcis ét ad gangen.

"Min database er forsvundet" er næsten altid et projekt der aldrig blev sat — dataene er
der stadig, men sessionen peger et andet sted hen.

`set_current` opretter projektet hvis det ikke findes.

In [ ]:
import bw2data as bd

PROJECT = 'workshop'          # skift til dit eget navn hvis du vil

bd.projects.set_current(PROJECT)
print('Projekt:', bd.projects.current)
print('Databaser:', list(bd.databases) or '(ingen endnu)')

## 3. Er ecoinvent der allerede?

Cellen leder efter en ecoinvent-database i projektet. Den kigger også i dine **andre**
projekter — hvis du har hentet ecoinvent før i et andet projekt, siger den til, så du
kan kopiere i stedet for at downloade forfra.

Navnene varierer mellem versioner (`ecoinvent-3.11-consequential`, `biosphere3`,
`ecoinvent-3.11-biosphere`), så cellen leder efter mønstre frem for faste navne.

In [ ]:
def find_ecoinvent(dbs):
    """Navne der ligner en ecoinvent-database, undtagen biosfæren."""
    return [d for d in dbs
            if 'ecoinvent' in d.lower() and 'biosphere' not in d.lower()]

def find_biosphere(dbs):
    return [d for d in dbs
            if 'biosphere' in d.lower()]

here = list(bd.databases)
ei_here = find_ecoinvent(here)
bio_here = find_biosphere(here)

if ei_here:
    print(f"✓ ecoinvent findes i '{PROJECT}':")
    for d in ei_here:
        print('   ', d, f'({len(bd.Database(d))} aktiviteter)')
    for d in bio_here:
        print('   ', d, f'({len(bd.Database(d))} flows)')
    print('\n→ Spring trin 4 over. Gå til trin 5.')
else:
    print(f"Ingen ecoinvent i '{PROJECT}'.\n")
    # Kig i de andre projekter, så vi ikke downloader unødigt
    others = {}
    for p in bd.projects:
        if p.name == PROJECT:
            continue
        bd.projects.set_current(p.name)
        found = find_ecoinvent(list(bd.databases))
        if found:
            others[p.name] = found
    bd.projects.set_current(PROJECT)     # tilbage hvor vi kom fra

    if others:
        print('Men ecoinvent findes i andre projekter på denne maskine:')
        for name, dbs in others.items():
            print(f'   {name}: {dbs}')
        print('\n→ Du kan kopiere i stedet for at downloade (se trin 4b),')
        print('  eller bare arbejde videre i det projekt.')
    else:
        print('→ ecoinvent skal downloades. Kør trin 4.')

## 4. Download ecoinvent — kun hvis trin 3 sagde det

**Sprang trin 3 den over? Så spring denne over.**

Cellen spørger om dit brugernavn og kodeord, når du kører den. Kodeordet skrives ikke
ind i notebooken og gemmes ikke nogen steder — så du kan dele filen bagefter uden at
dele dit login.

### Vælg systemmodel — et metodevalg, ikke en indstilling

Systemmodellen er en del af databasens navn, så hver eneste kobling i din model peger på
ét bestemt valg. At skifte bagefter betyder at hente forfra **og** skrive alle koblinger
om. Vælg rigtigt nu.

| Model | Bruges til |
|---|---|
| `cutoff` | De fleste attributionelle studier — det almindelige valg |
| `consequential` | Konsekvens-LCA |
| `apos` | Allocation at the point of substitution |

Ved du ikke hvilken dit projekt kræver, så spørg din vejleder eller se studieprotokollen.
Undervisningsmateriale viser ofte `consequential` fordi et bestemt kursus valgte det —
det er ikke en standard.

### Det tager 10-30 minutter, og der er ingen statuslinje

> Cellen viser `[*]` og ser fuldstændig død ud. **Den er ikke gået i stå.** Lad den køre.
>
> Afbryder du, kan du stå med en halvt importeret database der fejler forvirrende
> bagefter. Den hurtigste udvej er da at slette projektet og starte forfra.

Hav plads på disken: hvert ecoinvent-projekt fylder adskillige GB.

In [ ]:
import getpass
import bw2io as bi

VERSION = '3.11'              # streng, ikke tal
SYSTEM_MODEL = 'cutoff'       # 'cutoff' / 'consequential' / 'apos'

if find_ecoinvent(list(bd.databases)):
    print('ecoinvent er der allerede — downloader ikke. Gå til trin 5.')
else:
    print(f'Henter ecoinvent {VERSION} {SYSTEM_MODEL} til projektet "{PROJECT}".')
    print('Log ind med din ecoinvent.org-konto.\n')

    username = input('ecoinvent brugernavn: ')
    password = getpass.getpass('ecoinvent kodeord (vises ikke): ')

    print('\nDownloader. Dette tager 10-30 minutter uden statuslinje.')
    print('Afbryd ikke cellen.\n')

    bi.import_ecoinvent_release(
        version=VERSION,
        system_model=SYSTEM_MODEL,
        username=username,
        password=password,
    )

    del password      # ud af hukommelsen igen
    print('\n✓ Færdig. Databaser nu:', list(bd.databases))

### 4b. Alternativ: kopiér fra et andet projekt

Sagde trin 3 at ecoinvent findes i et *andet* projekt, kan du kopiere det projekt i
stedet for at downloade igen. Det tager sekunder frem for en halv time.

Bemærk at kopien fylder lige så meget som originalen — ecoinvent-projekter deler ikke
data indbyrdes.

Fjern `#` og sæt navnet på kildeprojektet ind:

In [ ]:
# bd.projects.set_current('navn-på-projekt-der-har-ecoinvent')
# bd.projects.copy_project('workshop-kopi', switch=True)
# print(bd.projects.current, list(bd.databases))

## 5. Virker det? — tjek størrelsesordener

Et andet spørgsmål end "kører Brightway". Her tjekker vi at ecoinvent kom **helt** ind.

Det er værd at gøre nu, fordi en halvt gennemført import ser fin ud på overfladen og
først fejler uger senere — som en score på nul eller en `UnknownObject` midt i noget
andet.

Tallene varierer med version og systemmodel, så betragt dem som størrelsesordener.
Det afgørende er at **intet er nul eller encifret**.

In [ ]:
dbs = list(bd.databases)
ei_found = find_ecoinvent(dbs)
bio_found = find_biosphere(dbs)

if not ei_found or not bio_found:
    print('⚠ Der er ingen ecoinvent i dette projekt endnu.')
    print(f'  Projekt: {bd.projects.current}')
    print(f'  Databaser: {dbs or "(ingen)"}')
    print('\n  Gå tilbage til trin 3 og 4. Kør ikke videre herfra.')
else:
    ei_name = ei_found[0]
    bio_name = bio_found[0]

    n_act = len(bd.Database(ei_name))
    n_bio = len(bd.Database(bio_name))
    n_met = len(bd.methods)

    print(f'{ei_name:40s} {n_act:>7,} aktiviteter   (forvent titusinder)')
    print(f'{bio_name:40s} {n_bio:>7,} flows         (forvent tusinder)')
    print(f'{"LCIA-metoder":40s} {n_met:>7,} metoder       (forvent hundreder+)')

    if min(n_act, n_bio, n_met) < 10:
        print('\n⚠ Noget mangler. Importen blev ikke færdig.')
        print('  Slet projektet og start forfra frem for at bygge videre på det:')
        print(f"  bd.projects.delete_project('{PROJECT}', delete_dir=True)")
    else:
        print('\n✓ Ser rigtigt ud.')

## 6. Én rigtig beregning fra ende til anden

Den endelige prøve. Den bruger databaserne, koblingerne mellem dem og metoderne på én
gang — virker denne, virker hele stakken.

Vi beregner klimaaftrykket af 1 kWh dansk elektricitet.

Bemærk at vi **slår metodenavnet op** frem for at skrive det fra hukommelsen. Nøglerne
ændrer sig mellem ecoinvent-udgivelser, så en hardkodet nøgle knækker ved næste version.

In [ ]:
import bw2calc as bc

# Slå metoden op frem for at skrive nøglen fra hukommelsen — nøglerne
# ændrer sig mellem udgivelser.
#
# Vi vælger IPCC 2021 eksplicit. Tag ikke bare den første på listen:
# metodefamilierne (IPCC, EF, CML, ReCiPe) giver forskellige tal, og
# hvilken du bruger er et valg der skal stå klart i din rapport.
climate = [m for m in bd.methods
           if 'IPCC 2021' in str(m)
           and 'climate change' in str(m).lower()
           and 'GWP100' in str(m)
           and 'no LT' not in str(m)
           and 'SLCFs' not in str(m)]

if not climate:
    print('⚠ Ingen IPCC 2021-metoder fundet.')
    print('  Enten er ecoinvent ikke importeret, eller din version')
    print('  navngiver metoderne anderledes. Se hvad der findes med:')
    print("  [m for m in bd.methods if 'climate' in str(m).lower()][:10]")
    method_key = None
else:
    print('IPCC 2021 GWP100-metoder i dette projekt:')
    for m in climate:
        print('   ', m)

    # Den korteste nøgle er hovedindikatoren — ikke underkategorier
    # som 'climate change: fossil' eller ': biogenic'
    method_key = min(climate, key=lambda m: len(str(m)))
    print('\nBruger:', method_key)

In [ ]:
if method_key is None:
    print('⚠ Ingen metode valgt — kør cellen ovenfor først.')
else:
    ei = bd.Database(ei_name)

    hits = [a for a in ei
            if a['name'] == 'market for electricity, low voltage'
            and a['location'] == 'DK']

    print(len(hits), 'match')     # forvent præcis 1

    if not hits:
        print('\n⚠ Aktiviteten blev ikke fundet.')
        print('  Navnene varierer lidt mellem ecoinvent-versioner.')
        print('  Se hvad der findes for DK:')
        print("  sorted({a['name'] for a in ei if a['location']=='DK'"
              " and 'electricity' in a['name']})[:10]")
    else:
        act = hits[0]
        print(act['name'], '|', act['location'], '|', act['unit'])

        lca = bc.LCA({act: 1}, method_key)
        lca.lci()      # løs inventaret: hvad udledes i alt
        lca.lcia()     # vægt udledningerne med metoden

        print(f'\n{lca.score:.4f} {bd.methods[method_key]["unit"]} per kWh')

**Et plausibelt tal forskelligt fra nul betyder at alt virker.**

- Får du præcis `0.0`, er noget ikke koblet sammen.
- Fejler `.lci()`, mangler biosfære-databasen typisk, eller den hedder noget andet.

### Hvad er et plausibelt tal?

Det afhænger stærkt af systemmodellen, og forskellen er lærerig:

| Systemmodel | Dansk lavspændings-el, IPCC 2021 GWP100 |
|---|---|
| `consequential` | omkring **0,04** kg CO2-eq/kWh |
| `cutoff` | væsentligt højere — typisk **0,1–0,2** |

Hvorfor? I `consequential` er den marginale danske el domineret af vindkraft — altså
hvad der *faktisk* bygges mere af, når efterspørgslen stiger. I `cutoff` ser du
gennemsnittet af den nuværende blanding, som stadig rummer fossil kraft.

Samme spørgsmål, to forskellige svar, fordi de stiller spørgsmålet forskelligt. Det er
ikke en fejl i data — det er hele pointen med at vælge systemmodel bevidst.

---

## To advarsler du kan ignorere

Rød tekst i Jupyter er ikke altid en fejl:

- **scikit-umfpack** — beregninger er en anelse langsommere uden. Intet er i stykker.
- **"Not able to determine geocollections"** — handler om regionaliseret LCA, som du
  ikke laver. Importen lykkedes.

## Hvis noget går galt

| Symptom | Sandsynlig årsag |
|---|---|
| Login afvist, men kodeordet er rigtigt | Aftalen ikke accepteret på hjemmesiden — se toppen |
| Aldrig logget ind i browseren | Samme — gå på ecoinvent.org først |
| Virker for en medstuderende, ikke dig | I har licens til forskellige versioner |
| `version` eller `system_model` afvist | Sendt som tal i stedet for streng, eller ikke dækket af licensen |
| Cellen kører i en evighed | Normalt. 10-30 min. Afbryd ikke |
| Importen fejlede, nu opfører den sig sært | Halvt importeret projekt — slet det og start forfra |
| `Database ... is locked` | To notebooks åbne på samme projekt. Luk de andre |
| `IndexError` i trin 5 eller 6 | Ingen ecoinvent i projektet — kør trin 4 først |

**Tommelfingerregel: én notebook per projekt ad gangen.**

Sidder du fast, så spørg assistenten og indsæt hele fejlbeskeden — også de dele der ser
irrelevante ud.

---

## Videre

ecoinvent er nu din **baggrund**. Næste skridt er at bygge din egen **forgrund** — dit
eget produktsystem — og koble den til.

Spørg assistenten: *"Hjælp mig med at bygge et produktsystem for ..."*